# Week 5 Day 2: LangChain Travel Concierge Agent

This notebook rebuilds the Day 1 raw Python agent (`agent_gateway.py`) using LangChain,
themed as a Travel Agent that helps a user compare cities on weather, budget, and
cost of living.

Version note: LangChain 1.x removed `create_tool_calling_agent` and `AgentExecutor`
in favor of a LangGraph based `create_agent`. Since this assignment asks specifically
for `create_tool_calling_agent` / `AgentExecutor`, this notebook pins:

```
langchain==0.3.27
langchain-openai==0.3.28
langchain-core==0.3.72
```

Tools in this notebook:
1. `calculator`, reused from Day 1 (math.js API)
2. `get_weather`, reused from Day 1 (Open-Meteo API), expanded to return more fields
3. `get_city_cost_of_living`, new, reads a real cost of living dataset from a local CSV
4. `convert_currency`, new, calls a real live free API (Frankfurter, European Central Bank rates) with no key required


## Setup

Install packages (run once):

```
pip install langchain==0.3.27 langchain-openai==0.3.28 langchain-core==0.3.72 python-dotenv requests
```



In [18]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://llm.netixsol.com/v1"
MODEL_NAME = "smart"
GATEWAY_API_KEY = os.environ.get("GATEWAY_API_KEY")

if not GATEWAY_API_KEY:
    print("No GATEWAY_API_KEY set. Set it with")
else:
    print("key found")


key found


## Task 1: Core concepts and the model wrapper

Mapping from `agent_gateway.py` to LangChain:

| Day 1 raw Python | LangChain |
|---|---|
| `OpenAI(base_url=..., api_key=...)` | `ChatOpenAI(base_url=..., api_key=...)` |
| `TOOLS` list of hand written JSON schemas | `@tool` decorated Python functions |
| `TOOL_FUNCTIONS` dict and `execute_tool()` | handled inside `AgentExecutor` |
| the manual `for step in range(...)` loop | `AgentExecutor.invoke()` |
| the `messages` list built by hand | `RunnableWithMessageHistory` |
| `[REASON]/[ACT]/[OBSERVE]` print statements | `verbose=True` |

Temperature: set to `0.2` here. Low but not zero, since the agent should stay
consistent when picking tools and reading numbers, but the final recommendation
sentence can have a little natural variation in phrasing. 
LCEL pipe (`|`) explanation: `prompt | model | parser` builds a `RunnableSequence`.
Each component implements a shared `Runnable` interface with `.invoke()`. The `|`
operator overrides Python's bitwise-or so that `a | b` returns a new Runnable whose
`.invoke(x)` calls `b.invoke(a.invoke(x))`. So it is function composition with a
consistent interface, not special magic syntax, just an operator overload chaining
callables together.


In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(
    base_url=BASE_URL,
    api_key=GATEWAY_API_KEY,
    model=MODEL_NAME,
    temperature=0.2,
    max_tokens=1000,
)

# Minimal LCEL chain: prompt -> model -> string parser
basic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise travel assistant."),
    ("human", "{question}")
])

basic_chain = basic_prompt | model | StrOutputParser()

print(basic_chain.invoke({"question": "In one sentence, what is Reykjavik known for?"}))


Re Reykjavik is renowned for its vibrant blend of historic Viking heritage, cutting‑edge design, and a thriving nightlife centered around its colorful waterfront and geothermal hot‑spring culture.


## Task 2: Tools

Docstrings matter here. LangChain reads the function's docstring (plus type hints)
and sends that text to the model as the tool description. The model never sees your
Python code, it only sees the docstring and the argument schema. A vague docstring
means the model will guess wrong about when to call the tool or what to pass it, the
same way a vague variable name confuses a human reading code later.

Real data sources used:
- `get_city_cost_of_living` reads `city_cost_data.csv`, a small dataset built from
  Numbeo's 2026 Cost of Living Index (real published index numbers, not invented).
  New York is the baseline city at index 100, all other cities are relative to it.
- `convert_currency` calls the Frankfurter API (`api.frankfurter.dev`), a real, free,
  no key required exchange rate service backed by European Central Bank reference
  rates. This is the live external call for the "touches real data" requirement.


In [4]:
from langchain_core.tools import tool

COST_DATA = pd.read_csv("city_cost_data.csv")


def run_calculator(expression: str) -> str:
    try:
        response = requests.get(
            "https://api.mathjs.org/v4/", params={"expr": expression}, timeout=5
        )
    except requests.exceptions.RequestException as e:
        return f"ERROR: could not reach math.js API: {e}"
    if response.status_code != 200:
        return f"ERROR: math.js API returned {response.status_code}: {response.text.strip()}"
    return response.text.strip()


@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression using the math.js public API (real external service).
    Supports standard arithmetic (+, -, *, /, ^, parentheses) and functions like sqrt(),
    sin(), log(). Convert word problems into a plain expression first.
    Example: 'sqrt(16) + 2^3' or '45 * 3'
    """
    return run_calculator(expression)


@tool
def get_weather(city: str) -> str:
    """Look up current, real weather for a named city using the Open-Meteo API
    (free, live data, no key needed). Give a plain city name, the tool geocodes it
    internally. Returns temperature in Celsius, windspeed, and a simple sky
    condition. Returns an explicit error string if the city cannot be found.
    """
    try:
        geo_resp = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=5,
        )
        geo_resp.raise_for_status()
        geo_data = geo_resp.json()
    except requests.exceptions.RequestException as e:
        return f"ERROR: could not reach geocoding API: {e}"

    results = geo_data.get("results")
    if not results:
        return f"ERROR: no location found for city '{city}'"

    lat = results[0]["latitude"]
    lon = results[0]["longitude"]
    resolved_name = results[0].get("name", city)

    try:
        weather_resp = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lat,
                "longitude": lon,
                "current_weather": "true",
                "hourly": "relative_humidity_2m",
            },
            timeout=5,
        )
        weather_resp.raise_for_status()
        weather_data = weather_resp.json()
    except requests.exceptions.RequestException as e:
        return f"ERROR: could not reach forecast API: {e}"

    current = weather_data.get("current_weather")
    if not current:
        return f"ERROR: forecast API returned no current_weather for '{city}'"

    humidity = None
    try:
        hourly = weather_data.get("hourly", {})
        humidity_list = hourly.get("relative_humidity_2m", [])
        if humidity_list:
            humidity = humidity_list[0]
    except Exception:
        humidity = None

    return json.dumps({
        "city": resolved_name,
        "temp_c": current.get("temperature"),
        "windspeed_kmh": current.get("windspeed"),
        "humidity_pct": humidity,
        "weather_code": current.get("weathercode"),
        "is_day": current.get("is_day"),
    })


@tool
def get_city_cost_of_living(city: str) -> str:
    """Look up real cost of living data for a city..."""
    match = COST_DATA[COST_DATA["city"].str.lower() == city.strip().lower()]
    if match.empty:
        available = ", ".join(sorted(COST_DATA["city"].tolist()))
        return f"ERROR: '{city}' not found in cost of living dataset. Available cities: {available}"
    row = match.iloc[0]
    return json.dumps({
        "city": str(row["city"]),
        "country": str(row["country"]),
        "cost_of_living_index": float(row["cost_of_living_index"]),
        "rent_index": float(row["rent_index"]),
        "groceries_index": float(row["groceries_index"]),
        "restaurant_price_index": float(row["restaurant_price_index"]),
        "local_purchasing_power_index": float(row["local_purchasing_power_index"]),
        "avg_meal_inexpensive_usd": float(row["avg_meal_inexpensive_usd"]),
        "avg_hostel_night_usd": float(row["avg_hostel_night_usd"]),
    })


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between two currencies using live European Central Bank
    reference rates via the free Frankfurter API (no key required). Currency codes
    are 3 letter ISO codes, for example USD, EUR, JPY, GBP, ISK. Returns the
    converted amount and the exchange rate used.
    """
    try:
        resp = requests.get(
            "https://api.frankfurter.dev/v1/latest",
            params={"amount": amount, "base": from_currency.upper(), "symbols": to_currency.upper()},
            timeout=5,
        )
        resp.raise_for_status()
        data = resp.json()
    except requests.exceptions.RequestException as e:
        return f"ERROR: could not reach currency API: {e}"

    rates = data.get("rates", {})
    converted = rates.get(to_currency.upper())
    if converted is None:
        return f"ERROR: could not convert {from_currency} to {to_currency}, check currency codes"

    return json.dumps({
        "amount": amount,
        "from_currency": from_currency.upper(),
        "to_currency": to_currency.upper(),
        "converted_amount": round(converted, 2),
        "date": data.get("date"),
    })


TOOLS = [calculator, get_weather, get_city_cost_of_living, convert_currency]


## Task 3: Building the agent

`create_tool_calling_agent` wires the model, the tool list, and a prompt template
together into a runnable agent. `AgentExecutor` then runs the actual loop: send
messages, check if the model asked for a tool call, run the tool, feed the result
back, repeat until the model returns a final answer or `max_iterations` is hit.

`verbose=True` prints the same kind of Thought / Action / Observation trace your
Day 1 print statements produced by hand, except LangChain generates it for you from
inside the loop you no longer see directly.

`handle_parsing_errors=True` tells the executor to catch cases where the model's
output cannot be parsed as a valid tool call or final answer, and feed the parsing
error back to the model as an observation instead of crashing the whole run. This is
the LangChain equivalent of the `try/except json.JSONDecodeError` block in my Day 1
`run_agent()` function.


In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor

SYSTEM_PROMPT = """You are a Travel Concierge Bot. You help users compare cities on
weather, cost of living, and budget, using your tools rather than guessing numbers.
Always call a tool for any factual weather, cost, or currency figure, never estimate
these yourself. Keep answers concise and practical for a traveler deciding between
cities."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(model, TOOLS, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=6,
)


result = agent_executor.invoke({"input": "What's the weather in Tokyo and Reykjavik, and which is warmer right now?"})
print(result["output"])




> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Tokyo'}`


{"city": "Tokyo", "temp_c": 31.4, "windspeed_kmh": 4.5, "humidity_pct": 80, "weather_code": 0, "is_day": 1}
Invoking: `get_weather` with `{'city': 'Reykjavik'}`


{"city": "Reykjavik", "temp_c": 10.8, "windspeed_kmh": 7.2, "humidity_pct": 95, "weather_code": 45, "is_day": 1}- **Tokyo:** 31.4 °C, light breeze, humid.  
- **Reykjavik:** 10.8 °C, moderate breeze, very humid.

**Tokyo is considerably warmer than Reykjavik right now.**

> Finished chain.
- **Tokyo:** 31.4 °C, light breeze, humid.  
- **Reykjavik:** 10.8 °C, moderate breeze, very humid.

**Tokyo is considerably warmer than Reykjavik right now.**


In [17]:
result = agent_executor.invoke({"input": "If I have $500 USD, how much is that in Icelandic krona?"})
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `convert_currency` with `{'amount': 500, 'from_currency': 'USD', 'to_currency': 'ISK'}`


{"amount": 500.0, "from_currency": "USD", "to_currency": "ISK", "converted_amount": 62664, "date": "2026-07-20"}$500 USD is approximately **62,664 ISK** (Icelandic króna) based on the latest exchange rate.

> Finished chain.
$500 USD is approximately **62,664 ISK** (Icelandic króna) based on the latest exchange rate.


### Annotated trace

Copy the printed trace here and label each stage, for example:

Invoking: `get_weather` with `{'city': 'Tokyo'}`                          <- ACT (tool call 1)

{"city": "Tokyo", "temp_c": 31.4, "windspeed_kmh": 4.5,                    <- OBSERVE (tool result 1)
 "humidity_pct": 80, "weather_code": 0, "is_day": 1}

Invoking: `get_weather` with `{'city': 'Reykjavik'}`                       <- ACT (tool call 2)

{"city": "Reykjavik", "temp_c": 10.8, "windspeed_kmh": 7.2,                <- OBSERVE (tool result 2)
 "humidity_pct": 95, "weather_code": 45, "is_day": 1}

Tokyo: 31.4 C, light breeze, humid.                                        <- REASON
Reykjavik: 10.8 C, moderate breeze, very humid.                            <- REASON
Tokyo is considerably warmer than Reykjavik right now.                     <- final answer

> Finished chain.


Compare to the Day 1 `agent_gateway.py` log: same three stage pattern (act, observe,
reason), but in Day 1 you could see the exact `messages` list being built turn by
turn, including the raw `tool_call_id` bookkeeping. Here that bookkeeping happens
inside `AgentExecutor`, `verbose=True` shows you the tool calls and results but not
the literal message objects being appended behind the scenes.


## Task 4: Memory

`RunnableWithMessageHistory` wraps the agent executor and automatically loads and
saves a message history per `session_id`, so a follow up question like "compare it
to Y" can resolve "it" from an earlier turn without you manually re-appending
messages to a list, which is what the Day 1 `messages.append(...)` calls were doing
by hand.


In [6]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

_session_store = {}

def get_session_history(session_id: str):
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]

agent_with_memory = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

config = {"configurable": {"session_id": "trip-planner-demo"}}

# Three turn scenario, matches the Task 4 spec:
turn1 = agent_with_memory.invoke({"input": "What's the cost of living in Tokyo?"}, config=config)
turn2 = agent_with_memory.invoke({"input": "Now compare it to Reykjavik."}, config=config)
turn3 = agent_with_memory.invoke({"input": "Which one should I recommend to a budget-conscious backpacker?"}, config=config)
print(turn3["output"])




> Entering new AgentExecutor chain...

Invoking: `get_city_cost_of_living` with `{'city': 'Tokyo'}`


{"city": "Tokyo", "country": "Japan", "cost_of_living_index": 72.4, "rent_index": 38.9, "groceries_index": 68.1, "restaurant_price_index": 58.7, "local_purchasing_power_index": 88.2, "avg_meal_inexpensive_usd": 7.0, "avg_hostel_night_usd": 35.0}**Tokyo – Cost of Living Snapshot (USD)**  

| Metric | Value |
|--------|-------|
| Cost‑of‑Living Index* | **72.4** |
| Rent Index* | **38.9** |
| Groceries Index* | **68.1** |
| Restaurant Price Index* | **58.7** |
| Local Purchasing Power Index* | **88.2** |
| Avg. inexpensive meal (restaurant) | **≈ $7.00** |
| Avg. hostel/night | **≈ $35.00** |

\*Indices are relative to New York City (set at 100). A higher purchasing‑power index means residents can afford more with their income.

**What this means for a traveler**

- **Accommodation:** Hostels average about $35/night; mid‑range hotels will be considerably higher (often $80‑$150+).  
- *

## Task 5: Structured output and error handling

Structured output: rather than parsing the final sentence with regex, `with_structured_output`
tells the model to return data matching a Pydantic schema directly. This is the
cleanest way to get a machine readable answer out of an LLM call.

Error handling: each tool already returns an explicit `ERROR: ...` string instead of
raising, matching the Day 1 pattern, so the agent sees the failure as a normal
observation and can recover, for example by trying a different city name or telling
the user the data was unavailable. To also protect against a tool that raises a real
Python exception, `@tool` functions can set `handle_tool_error=True`, which tells
LangChain to catch the exception and hand the model a text description of it instead
of crashing the whole `AgentExecutor.invoke()` call.


In [8]:
from pydantic import BaseModel, Field
from typing import Literal


class TripRecommendation(BaseModel):
    destination: str = Field(description="Recommended city")
    budget_tier: Literal["budget", "mid-range", "luxury"] = Field(
        description="Overall budget tier of the recommended city"
    )
    weather_summary: str = Field(description="One sentence summary of current weather")
    cost_summary: str = Field(description="One sentence summary of cost of living vs alternatives")
    verdict: str = Field(description="Final recommendation sentence for the traveler")


structured_model = model.with_structured_output(TripRecommendation)

structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the travel comparison below into the required structured format."),
    ("human", "{comparison_text}"),
])

structured_chain = structured_prompt | structured_model


recommendation = structured_chain.invoke({"comparison_text": turn3["output"]})
print(recommendation.model_dump_json(indent=2))


{
  "destination": "Tokyo",
  "budget_tier": "budget",
  "weather_summary": "Tokyo currently experiences mild temperatures with occasional rain.",
  "cost_summary": "Tokyo’s cost of living is roughly 25% cheaper than Reykjavík, with hostels around $35 versus $45 and meals about $7 versus $22.",
  "verdict": "For a tight‑budget backpacker, Tokyo is the clear choice."
}


In [14]:
import random
from langchain_core.tools import ToolException

@tool
def flaky_currency_check(currency_code: str) -> str:
    """Check whether a currency code is actively supported. This tool
    intentionally fails about half the time to demonstrate error recovery,
    it is not a real production tool.
    """
    if random.random() < 0.5:
        raise ToolException(f"temporary lookup failure for currency {currency_code}")
    return f"{currency_code.upper()} is supported."

flaky_currency_check.handle_tool_error = True

In [15]:
for i in range(5):
    print(flaky_currency_check.run({"currency_code": "ISK"}))

temporary lookup failure for currency ISK
ISK is supported.
ISK is supported.
temporary lookup failure for currency ISK
temporary lookup failure for currency ISK


## Test cell: unknown city, to confirm graceful failure end to end


In [16]:
result = agent_executor.invoke({"input": "What's the cost of living in Atlantis?"})
print(result["output"])




> Entering new AgentExecutor chain...

Invoking: `get_city_cost_of_living` with `{'city': 'Atlantis'}`


ERROR: 'Atlantis' not found in cost of living dataset. Available cities: Bali, Bangkok, Berlin, Cape Town, Lisbon, London, Mexico City, New York, Reykjavik, Singapore, Tokyo, ZurichI’m sorry—I couldn’t find any cost‑of‑living data for “Atlantis.” It isn’t in the available dataset (which includes cities such as Bali, Bangkok, Berlin, Cape Town, Lisbon, London, Mexico City, New York, Reykjavik, Singapore, Tokyo, and Zurich). 

If you have another city in mind, let me know and I can pull the latest figures for you!

> Finished chain.
I’m sorry—I couldn’t find any cost‑of‑living data for “Atlantis.” It isn’t in the available dataset (which includes cities such as Bali, Bangkok, Berlin, Cape Town, Lisbon, London, Mexico City, New York, Reykjavik, Singapore, Tokyo, and Zurich). 

If you have another city in mind, let me know and I can pull the latest figures for you!


## Summary

What LangChain made easier: the tool dispatch loop, message history bookkeeping,
and JSON parsing of tool call arguments are all handled internally instead of
hand written, and swapping the model provider only means changing the `ChatOpenAI`
constructor arguments, not the agent logic.

What felt like magic or added complexity: `verbose=True` shows tool calls and
results but not the exact message list being assembled turn by turn, so debugging a
malformed tool call is less direct than reading your own `messages.append(...)`
calls from Day 1. The framework's own agent API also changed between major versions
(`create_tool_calling_agent` existed in 0.3.x, removed in 1.x in favor of
`create_agent`), which is worth noting as a real maintenance cost of depending on a
fast moving framework.
